In [14]:
from langchain_classic.document_loaders import TextLoader,DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model
from langchain_classic.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser



In [15]:
loader = DirectoryLoader(r'C:\Users\mendh\Langchain-Langgraph\0-Dataingestion-parsing\data\text_files', glob='**/*.txt', show_progress=True,loader_cls=TextLoader)
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents)
print(f"Number of documents: {len(docs)}")

100%|██████████| 4/4 [00:00<00:00, 1848.32it/s]

Number of documents: 28


In [16]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)
retriver = vectorstore.as_retriever(search_kwargs={"k": 5})
retriver

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000017402C13D90>, search_kwargs={'k': 5})

In [17]:
import os
from dotenv import load_dotenv
load_dotenv()
llm = init_chat_model(model_provider="groq", model="llama-3.1-8b-instant")
llm.invoke("Hello, how are you?")

AIMessage(content="I'm functioning properly, thank you for asking. I'm a large language model, so I don't have feelings or emotions like humans do, but I'm here to help answer any questions you might have or provide information on a wide range of topics. How can I assist you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 41, 'total_tokens': 100, 'completion_time': 0.295362139, 'completion_tokens_details': None, 'prompt_time': 0.003456172, 'prompt_tokens_details': None, 'queue_time': 0.047717173, 'total_time': 0.298818311}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d4854-09ee-7273-bb8e-b95f781b15f9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 41, 'output_tokens': 59, 'total_tokens': 100})

In [22]:
query  = PromptTemplate.from_template("""you are a helpful assistant. Expand the following query to improve document retrieval by adding synonms,technical terms, and useful context
original query:{query}:

Expanded query:""")

In [23]:
query_expansion_chain = query | llm | StrOutputParser()

In [25]:
print(query_expansion_chain.invoke({"query": "Langchain memory"}))

Here's an expanded query for improving document retrieval on "Langchain memory":

**Original Query:** Langchain memory

**Expanded Query:**

1. **Technical Terms:**
	* "Langchain memory" OR "memory-based Langchain" OR "Langchain knowledge graph"
	* "knowledge repository" OR "memory database" OR "cognitive memory storage"
2. **Synonyms:**
	* "Langchain" OR "Language Chain" OR "LC"
	* "memory" OR "knowledge storage" OR "cognitive memory"
3. **Useful Context:**
	* "AI-assisted knowledge management" OR "cognitive memory retrieval" OR "natural language processing (NLP) and memory"
	* "Langchain architecture" OR "memory-based Langchain components" OR "memory indexing and retrieval"
4. **Additional Keywords:**
	* "graph database" OR "knowledge graph database" OR "semantic memory"
	* "knowledge retrieval" OR "memory-based search" OR "cognitive search engine"
	* "AI knowledge base" OR "cognitive knowledge management" OR "knowledge management system (KMS)"
5. **Domain-Specific Terms:**
	* "natur